<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/T1/4_RecA_Classical_ML_Modeling_Q1_VALIDATION_AUDITED_FIXED_PLUS_DIAGNOSTICS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RecA Classical Machine Learning Modeling Workflow

**Publication-ready notebook** for constructing, optimizing, evaluating, and interpreting classical machine-learning models for **RecA inhibitor classification** using PaDEL molecular fingerprints/descriptors.

This notebook combines:

1. The RecA-specific full modeling workflow from the user's notebook.
2. The clearer SVM/RF/XGBoost modeling structure from the senior reference notebook.

The workflow is designed to be reproducible and thesis/manuscript-ready: stratified splitting, leakage-aware preprocessing, cross-validation, hyperparameter optimization, independent test-set evaluation, ROC/confusion-matrix visualization, model comparison, feature importance, optional SHAP interpretation, and export of all results.

## 1. Environment setup

Run this cell first. The notebook uses common scientific Python packages. Optional packages such as `xgboost` and `shap` are handled safely: if they are unavailable, the notebook will continue with the available models.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    make_scorer,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    StratifiedShuffleSplit,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    XGBClassifier = None
    HAS_XGBOOST = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    shap = None
    HAS_SHAP = False

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 10

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling_publication_ready"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

for folder in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("XGBoost available:", HAS_XGBOOST)
print("SHAP available:", HAS_SHAP)
print("Output directory:", OUTPUT_DIR.resolve())
from sklearn.utils import resample

## 2. Input dataset

Recommended input from the previous workflow:

- `outputs/feature_selection/03_recA_top_200_fscore.csv`, or
- `outputs/02_recA_modeling_matrix.csv`, or
- any RecA fingerprint/feature-selection matrix containing descriptor columns and a class label.

The target column can be named `class`, `bioactivity_class`, `Activity`, `label`, or `target`. The notebook automatically detects the target column and converts common active/inactive labels to binary values.

In [ ]:
# Change this path if your dataset has a different file name.
# Prefer the raw modeling matrix. Pre-selected feature files are kept only as fallback,
# because using a feature-selected file generated before train/test splitting can cause selection leakage.
CANDIDATE_INPUT_FILES = [
    PROJECT_DIR / "outputs" / "02_recA_modeling_matrix.csv",
    PROJECT_DIR / "02_recA_modeling_matrix.csv",
    PROJECT_DIR / "outputs" / "feature_selection" / "03_recA_low_variance_filtered.csv",
    PROJECT_DIR / "outputs" / "feature_selection" / "03_recA_top_200_fscore.csv",
    PROJECT_DIR / "03_recA_top_200_fscore.csv",
]

INPUT_FILE = next((p for p in CANDIDATE_INPUT_FILES if p.exists()), None)

if INPUT_FILE is None:
    raise FileNotFoundError(
        "No input dataset was found. Please place the RecA modeling matrix in one of these locations:\n"
        + "\n".join(str(p) for p in CANDIDATE_INPUT_FILES)
    )

print("Using input file:", INPUT_FILE)
df = pd.read_csv(INPUT_FILE)
print("Dataset shape:", df.shape)
df.head()

## 3. Data cleaning and target preparation

This section removes metadata columns from the feature matrix, keeps only numeric descriptors/fingerprints, removes missing/infinite values, and prepares the binary classification target.

In [ ]:
TARGET_CANDIDATES = ["class", "bioactivity_class", "Activity", "activity", "label", "target", "y"]

# Columns that must NOT be used as molecular descriptors because they are identifiers,
# assay metadata, or direct/near-direct activity information.
META_COLUMNS = [
    "Name", "Molecule ChEMBL ID", "chembl_id", "canonical_smiles", "SMILES", "smiles",
    "standard_type", "standard_value", "standard_value_nM", "standard_units",
    "pEC50", "pIC50", "pChEMBL", "pchembl_value",
    "EC50", "IC50", "Ki", "Kd",
    "assay_description", "target_organism", "target_pref_name"
]

# Pattern-based leakage guard. This removes any column whose name suggests that it
# contains target/activity/assay-derived values rather than structure descriptors.
LEAKAGE_PATTERNS = [
    "standard", "activity", "bioactivity", "pchembl", "pic50", "pec50",
    "ic50", "ec50", "ki", "kd", "label", "target", "class"
]


def detect_target_column(data: pd.DataFrame) -> str:
    for col in TARGET_CANDIDATES:
        if col in data.columns:
            return col
    raise ValueError(f"Target column not found. Expected one of: {TARGET_CANDIDATES}")


def encode_binary_target(y: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(y):
        unique_values = sorted(pd.Series(y.dropna().unique()).tolist())
        if set(unique_values).issubset({0, 1}):
            return y.astype(int)
        # For continuous activity values, use median split only as a fallback.
        threshold = y.median()
        print(f"Numeric non-binary target detected. Applying median split at {threshold:.4f}.")
        return (y >= threshold).astype(int)

    mapping = {
        "active": 1, "actives": 1, "1": 1, "yes": 1, "high": 1,
        "inactive": 0, "inactives": 0, "0": 0, "no": 0, "low": 0,
    }
    y_clean = y.astype(str).str.strip().str.lower().map(mapping)
    if y_clean.isna().any():
        unknown = sorted(y.astype(str)[y_clean.isna()].unique().tolist())
        raise ValueError(f"Unrecognized target labels: {unknown}")
    return y_clean.astype(int)


target_col = detect_target_column(df)
y = encode_binary_target(df[target_col])

feature_df = df.drop(columns=[target_col], errors="ignore")
feature_df = feature_df.drop(columns=[c for c in META_COLUMNS if c in feature_df.columns], errors="ignore")

# Remove potential leakage columns by name pattern.
potential_leakage_cols = [
    c for c in feature_df.columns
    if any(pattern in c.lower() for pattern in LEAKAGE_PATTERNS)
]
feature_df = feature_df.drop(columns=potential_leakage_cols, errors="ignore")

# Keep only numeric descriptor/fingerprint columns.
feature_df = feature_df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.dropna(axis=1, how="all").fillna(feature_df.median(numeric_only=True))

# Remove constant descriptors before splitting. This does not use the target.
constant_cols = [c for c in feature_df.columns if feature_df[c].nunique(dropna=False) <= 1]
X = feature_df.drop(columns=constant_cols, errors="ignore")

print("Detected target column:", target_col)
print("Removed constant columns:", len(constant_cols))
print("Removed potential leakage columns:", potential_leakage_cols)
print("Final feature matrix:", X.shape)
print("Class distribution:")
print(y.value_counts().rename({0: "inactive", 1: "active"}))

if y.nunique() != 2:
    raise ValueError("This notebook requires a binary classification target.")

# Final leakage audit
remaining_suspicious = [
    c for c in X.columns
    if any(pattern in c.lower() for pattern in LEAKAGE_PATTERNS)
]
if remaining_suspicious:
    raise ValueError(f"Potential leakage columns remain in X: {remaining_suspicious}")
else:
    print("Leakage audit passed: no activity/target-like columns remain in X.")


### Leakage-control note

Activity-derived columns such as `standard_value`, `standard_value_nM`, `pIC50`, `pEC50`, `pChEMBL`, `IC50`, and `EC50` are removed before modeling. Feature selection, scaling, and model training are performed inside scikit-learn `Pipeline` objects after the train/test split, which reduces data leakage risk.

**Important:** use the raw modeling matrix as input whenever possible. If an already pre-selected file such as `top_200_fscore.csv` was generated using the full dataset before splitting, that file can still introduce selection leakage.


## 4. Stratified train-test split

The train-test split follows the senior notebook style but uses RecA data and keeps class proportions balanced using `stratify=y`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True).round(3))
print("y_test distribution:\n", y_test.value_counts(normalize=True).round(3))

## 5. Evaluation functions

Metrics reported for manuscript/thesis tables:

- Accuracy
- Precision
- Recall/Sensitivity
- F1-score
- Matthews correlation coefficient (MCC)
- ROC-AUC

In [ ]:
def evaluate_predictions(y_true, y_pred, y_proba=None) -> dict:
    """Return QSAR classification metrics.

    Sensitivity is equivalent to recall for the active class.
    Specificity is the true-negative rate for the inactive class.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp + 1e-12)
        sensitivity = tp / (tp + fn + 1e-12)
    else:
        specificity = np.nan
        sensitivity = recall_score(y_true, y_pred, zero_division=0)

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1_score": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        metrics["ROC_AUC"] = roc_auc_score(y_true, y_proba)
    else:
        metrics["ROC_AUC"] = np.nan
    return metrics


def specificity_score_binary(y_true, y_pred) -> float:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape != (2, 2):
        return np.nan
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp + 1e-12)


specificity_scorer = make_scorer(specificity_score_binary)


def evaluate_cross_validation(estimator, X_data, y_data, cv) -> dict:
    scoring = {
        "Accuracy": "accuracy",
        "Precision": "precision",
        "Recall": "recall",
        "Sensitivity": "recall",
        "Specificity": specificity_scorer,
        "F1_score": "f1",
        "MCC": make_scorer(matthews_corrcoef),
        "ROC_AUC": "roc_auc",
    }
    cv_results = cross_validate(estimator, X_data, y_data, cv=cv, scoring=scoring, n_jobs=-1)
    return {
        metric: float(np.mean(cv_results[f"test_{metric}"]))
        for metric in scoring.keys()
    }


def metrics_to_frame(metrics: dict, model_name: str, dataset_name: str) -> pd.DataFrame:
    row = {"Model": model_name, "Dataset": dataset_name}
    row.update(metrics)
    return pd.DataFrame([row])


def get_positive_probability(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_data)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return None

## 6. Model pipelines and hyperparameter grids

The models follow the senior notebook structure: **SVM**, **Random Forest**, and **XGBoost**. ExtraTrees is also included as a strong tree-based benchmark commonly useful for sparse molecular fingerprint data.

To reduce data leakage, preprocessing and feature selection are placed inside the `Pipeline`, so cross-validation learns them only from the training folds.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# K is capped to avoid errors when the number of descriptors is smaller than 200.
K_BEST = min(200, X_train.shape[1])

model_spaces = {
    "SVM": {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=RANDOM_STATE)),
        ]),
        "params": {
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", 0.01, 0.1, 1],
            "model__kernel": ["rbf", "linear"],
        },
    },
    "RandomForest": {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")),
        ]),
        "params": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [None, 5, 10, 15],
            "model__min_samples_split": [2, 4],
            "model__min_samples_leaf": [1, 2],
        },
    },
    "ExtraTrees": {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("model", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")),
        ]),
        "params": {
            "model__n_estimators": [300, 500, 800],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 4],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", "log2", None],
        },
    },
}

if HAS_XGBOOST:
    model_spaces["XGBoost"] = {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "params": {
            "model__n_estimators": [100, 300, 500],
            "model__max_depth": [3, 4, 5],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        },
    }

print("Models to benchmark:", list(model_spaces.keys()))

## 7. Hyperparameter optimization and model evaluation

Each model is optimized using stratified 10-fold cross-validation on the training set only. The independent test set is used once for final evaluation.

In [ ]:
best_models = {}
all_metrics = []
best_params = {}

for model_name, spec in model_spaces.items():
    print(f"\n===== {model_name} =====")
    search = GridSearchCV(
        estimator=spec["pipeline"],
        param_grid=spec["params"],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0,
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_models[model_name] = best_model
    best_params[model_name] = search.best_params_

    cv_metrics = evaluate_cross_validation(best_model, X_train, y_train, cv=cv)
    y_pred = best_model.predict(X_test)
    y_proba = get_positive_probability(best_model, X_test)
    test_metrics = evaluate_predictions(y_test, y_pred, y_proba)

    all_metrics.append(metrics_to_frame(cv_metrics, model_name, "Train_10fold_CV"))
    all_metrics.append(metrics_to_frame(test_metrics, model_name, "Independent_Test"))

    print("Best parameters:", search.best_params_)
    print("Best CV ROC-AUC from GridSearch:", round(search.best_score_, 4))
    print("Independent test ROC-AUC:", round(test_metrics["ROC_AUC"], 4))

metrics_df = pd.concat(all_metrics, ignore_index=True)
metrics_df = metrics_df.sort_values(["Dataset", "ROC_AUC"], ascending=[True, False])
metrics_df.to_csv(OUTPUT_DIR / "04_recA_model_comparison_metrics.csv", index=False)

with open(OUTPUT_DIR / "04_recA_best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=2)

metrics_df.round(4)

## 8. Select the best model

The best model is selected based on independent test ROC-AUC. For small molecular datasets, also inspect MCC, F1-score, and the confusion matrix before making a final manuscript claim.

In [ ]:
test_table = metrics_df[metrics_df["Dataset"] == "Independent_Test"].copy()
best_model_name = test_table.sort_values("ROC_AUC", ascending=False).iloc[0]["Model"]
best_model = best_models[best_model_name]

print("Best model based on independent test ROC-AUC:", best_model_name)
display(test_table.sort_values("ROC_AUC", ascending=False).round(4))

joblib.dump(best_model, MODEL_DIR / f"04_recA_best_model_{best_model_name}.joblib")
print("Saved best model to:", MODEL_DIR / f"04_recA_best_model_{best_model_name}.joblib")

## 9. Confusion matrices

This section saves publication-ready confusion matrices for all models.

In [ ]:
for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Inactive", "Active"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{model_name} Confusion Matrix")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"04_confusion_matrix_{model_name}.png", bbox_inches="tight")
    plt.show()

## 10. ROC curves

ROC curves are generated for all optimized models and saved as high-resolution PNG files.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

roc_rows = []
for model_name, model in best_models.items():
    y_proba = get_positive_probability(model, X_test)
    if y_proba is None:
        continue
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    auc_value = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_value:.3f})")
    roc_rows.append(pd.DataFrame({"Model": model_name, "FPR": fpr, "TPR": tpr, "Threshold": thresholds}))

ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves for RecA Classification Models")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_roc_curves_all_models.png", bbox_inches="tight")
plt.show()

if roc_rows:
    pd.concat(roc_rows, ignore_index=True).to_csv(OUTPUT_DIR / "04_recA_roc_curve_coordinates.csv", index=False)

## 11. Q1-level diagnostic metrics table

This section explicitly reports the metrics commonly requested by QSAR reviewers: ROC-AUC, MCC, sensitivity, specificity, precision, recall, F1-score, and accuracy. The values are computed on the independent test set using the optimized models.

In [ ]:
q1_metric_rows = []

for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = get_positive_probability(model, X_test)
    metrics = evaluate_predictions(y_test, y_pred, y_proba)
    q1_metric_rows.append({"Model": model_name, **metrics})

q1_metrics_df = pd.DataFrame(q1_metric_rows).sort_values("ROC_AUC", ascending=False)
q1_metrics_df.to_csv(
    OUTPUT_DIR / "04_q1_test_metrics_accuracy_auc_mcc_sensitivity_specificity.csv",
    index=False,
)

q1_metrics_df

## 12. Bootstrap confidence intervals

A single holdout score can be unstable for QSAR datasets. Bootstrap confidence intervals provide uncertainty estimates for ROC-AUC, MCC, sensitivity, specificity, F1-score, and accuracy on the independent test set.

In [ ]:
def bootstrap_metric_ci(
    y_true,
    y_pred,
    y_proba=None,
    n_bootstrap=1000,
    confidence=0.95,
    random_state=RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_proba = None if y_proba is None else np.asarray(y_proba)

    metric_values = {
        "Accuracy": [],
        "MCC": [],
        "Sensitivity": [],
        "Specificity": [],
        "F1_score": [],
        "ROC_AUC": [],
    }

    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue

        proba_i = None if y_proba is None else y_proba[idx]
        m = evaluate_predictions(y_true[idx], y_pred[idx], proba_i)

        for key in metric_values:
            metric_values[key].append(m.get(key, np.nan))

    alpha = (1 - confidence) / 2
    rows = []
    for metric, values in metric_values.items():
        values = np.asarray(values, dtype=float)
        values = values[~np.isnan(values)]
        if len(values) == 0:
            rows.append({"Metric": metric, "Mean": np.nan, "CI_low": np.nan, "CI_high": np.nan})
        else:
            rows.append({
                "Metric": metric,
                "Mean": float(np.mean(values)),
                "CI_low": float(np.quantile(values, alpha)),
                "CI_high": float(np.quantile(values, 1 - alpha)),
            })
    return pd.DataFrame(rows)


bootstrap_ci_tables = []

for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = get_positive_probability(model, X_test)
    ci_df = bootstrap_metric_ci(y_test, y_pred, y_proba, n_bootstrap=1000)
    ci_df.insert(0, "Model", model_name)
    bootstrap_ci_tables.append(ci_df)

bootstrap_ci_df = pd.concat(bootstrap_ci_tables, ignore_index=True)
bootstrap_ci_df.to_csv(OUTPUT_DIR / "04_bootstrap_confidence_intervals.csv", index=False)

bootstrap_ci_df

## 13. Y-randomization test

Y-randomization checks whether high model performance could arise from chance correlation. The labels are randomly permuted repeatedly, and the optimized modeling pipeline is refitted on each randomized label vector. A valid QSAR model should substantially outperform the randomized-label distribution.

In [ ]:
def run_y_randomization(
    base_model,
    X_train,
    y_train,
    X_test,
    y_test,
    n_permutations=100,
    random_state=RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)
    rows = []

    for i in range(n_permutations):
        y_train_perm = pd.Series(rng.permutation(np.asarray(y_train)), index=y_train.index)
        model_perm = clone(base_model)
        model_perm.fit(X_train, y_train_perm)

        y_pred_perm = model_perm.predict(X_test)
        y_proba_perm = get_positive_probability(model_perm, X_test)

        metrics_perm = evaluate_predictions(y_test, y_pred_perm, y_proba_perm)
        rows.append({
            "Permutation": i + 1,
            **metrics_perm,
        })

    return pd.DataFrame(rows)


# Use the best-performing optimized model as the main Y-randomization target.
y_randomization_df = run_y_randomization(
    best_model,
    X_train,
    y_train,
    X_test,
    y_test,
    n_permutations=100,
)

original_auc = q1_metrics_df.loc[q1_metrics_df["Model"] == best_model_name, "ROC_AUC"].iloc[0]
random_auc_mean = y_randomization_df["ROC_AUC"].mean()
random_auc_std = y_randomization_df["ROC_AUC"].std()
p_value = (np.sum(y_randomization_df["ROC_AUC"] >= original_auc) + 1) / (len(y_randomization_df) + 1)

y_randomization_summary = pd.DataFrame([{
    "Best_Model": best_model_name,
    "Original_ROC_AUC": original_auc,
    "Randomized_ROC_AUC_mean": random_auc_mean,
    "Randomized_ROC_AUC_std": random_auc_std,
    "Empirical_p_value": p_value,
    "N_permutations": len(y_randomization_df),
}])

y_randomization_df.to_csv(OUTPUT_DIR / "04_y_randomization_runs.csv", index=False)
y_randomization_summary.to_csv(OUTPUT_DIR / "04_y_randomization_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
ax.hist(y_randomization_df["ROC_AUC"].dropna(), bins=20, alpha=0.75, edgecolor="black")
ax.axvline(original_auc, linestyle="--", linewidth=2, label=f"Original ROC-AUC = {original_auc:.3f}")
ax.set_xlabel("ROC-AUC after label randomization")
ax.set_ylabel("Frequency")
ax.set_title(f"Y-randomization test: {best_model_name}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"04_y_randomization_{best_model_name}.png", bbox_inches="tight")
plt.show()

y_randomization_summary

## 14. Applicability Domain using leverage approach

The applicability domain evaluates whether test compounds fall within the chemical descriptor space represented by the training set. Here, the warning leverage threshold is defined as \(h^* = 3(p+1)/n\), where \(p\) is the number of selected features and \(n\) is the number of training samples.

In [ ]:
def get_transformed_matrix_for_ad(pipeline, X_data):
    """Apply all preprocessing/feature-selection steps before the final classifier."""
    Xt = X_data.copy()
    for step_name, step in pipeline.steps[:-1]:
        Xt = step.transform(Xt)
    return np.asarray(Xt, dtype=float)


def compute_leverage_scores(X_train_transformed, X_test_transformed):
    Xtr = np.asarray(X_train_transformed, dtype=float)
    Xte = np.asarray(X_test_transformed, dtype=float)

    # Add intercept column.
    Xtr_aug = np.column_stack([np.ones(Xtr.shape[0]), Xtr])
    Xte_aug = np.column_stack([np.ones(Xte.shape[0]), Xte])

    xtx_inv = np.linalg.pinv(Xtr_aug.T @ Xtr_aug)
    leverage_train = np.sum((Xtr_aug @ xtx_inv) * Xtr_aug, axis=1)
    leverage_test = np.sum((Xte_aug @ xtx_inv) * Xte_aug, axis=1)

    p = Xtr.shape[1]
    n = Xtr.shape[0]
    warning_leverage = 3 * (p + 1) / n

    return leverage_train, leverage_test, warning_leverage


X_train_ad = get_transformed_matrix_for_ad(best_model, X_train)
X_test_ad = get_transformed_matrix_for_ad(best_model, X_test)

h_train, h_test, h_star = compute_leverage_scores(X_train_ad, X_test_ad)

test_pred = best_model.predict(X_test)
test_proba = get_positive_probability(best_model, X_test)
residuals = np.asarray(y_test) - np.asarray(test_proba if test_proba is not None else test_pred)
std_residuals = residuals / (np.std(residuals) + 1e-12)

ad_df = pd.DataFrame({
    "Leverage": h_test,
    "Standardized_residual": std_residuals,
    "Actual": np.asarray(y_test),
    "Predicted": np.asarray(test_pred),
    "Probability_active": test_proba if test_proba is not None else np.nan,
})
ad_df["Inside_AD"] = (ad_df["Leverage"] <= h_star) & (np.abs(ad_df["Standardized_residual"]) <= 3)

ad_df.to_csv(OUTPUT_DIR / f"04_applicability_domain_{best_model_name}.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
ax.scatter(ad_df["Leverage"], ad_df["Standardized_residual"], alpha=0.8)
ax.axvline(h_star, linestyle="--", linewidth=2, label=f"h* = {h_star:.3f}")
ax.axhline(3, linestyle="--", linewidth=1)
ax.axhline(-3, linestyle="--", linewidth=1)
ax.set_xlabel("Leverage")
ax.set_ylabel("Standardized residual")
ax.set_title(f"Applicability Domain: {best_model_name}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"04_applicability_domain_{best_model_name}.png", bbox_inches="tight")
plt.show()

pd.DataFrame([{
    "Model": best_model_name,
    "Warning_leverage_h_star": h_star,
    "Test_compounds_inside_AD": int(ad_df["Inside_AD"].sum()),
    "Total_test_compounds": len(ad_df),
    "Inside_AD_fraction": ad_df["Inside_AD"].mean(),
}])

## 15. External validation split

To strengthen the validation design, a separate external validation subset is held out before model development. The selected best pipeline is retrained on the model-development subset and evaluated on this external validation subset. This is a practical internal-external validation when a completely independent external dataset is unavailable.

### Additional internal holdout validation

This section creates an additional 10% stratified holdout from the same dataset. It is useful as a robustness check, but it should be reported as **additional internal validation**, not as a true external validation. A true external validation set must come from an independent source, assay, or temporal split.


In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=RANDOM_STATE)

dev_idx, external_idx = next(sss.split(X, y))
X_dev, X_external = X.iloc[dev_idx], X.iloc[external_idx]
y_dev, y_external = y.iloc[dev_idx], y.iloc[external_idx]

external_model = clone(best_model)
external_model.fit(X_dev, y_dev)

external_pred = external_model.predict(X_external)
external_proba = get_positive_probability(external_model, X_external)

external_metrics = evaluate_predictions(y_external, external_pred, external_proba)
external_validation_df = pd.DataFrame([{
    "Model": best_model_name,
    "Validation": "Additional_internal_holdout_10_percent",
    **external_metrics,
}])

external_validation_df.to_csv(OUTPUT_DIR / f"04_external_validation_{best_model_name}.csv", index=False)

fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
cm_ext = confusion_matrix(y_external, external_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm_ext, display_labels=["Inactive", "Active"])
disp.plot(ax=ax, values_format="d", colorbar=False)
ax.set_title(f"Additional Internal Holdout Confusion Matrix: {best_model_name}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"04_external_validation_confusion_matrix_{best_model_name}.png", bbox_inches="tight")
plt.show()

external_validation_df

## 11. Model comparison table

This table can be copied directly into the thesis/manuscript after checking formatting.

In [ ]:
comparison_pivot = metrics_df.pivot(index="Model", columns="Dataset", values=["Accuracy", "Precision", "Recall", "F1_score", "MCC", "ROC_AUC"])
comparison_pivot.to_csv(OUTPUT_DIR / "04_recA_model_comparison_pivot.csv")
comparison_pivot.round(4)

## 12. Selected features from the best pipeline

Because feature selection is inside the pipeline, this cell retrieves the actual selected descriptors/fingerprints used by the best model.

In [ ]:
def extract_selected_feature_names(pipeline: Pipeline, original_columns: pd.Index) -> list[str]:
    columns = np.array(original_columns)

    if "variance" in pipeline.named_steps:
        variance_mask = pipeline.named_steps["variance"].get_support()
        columns = columns[variance_mask]

    if "select" in pipeline.named_steps:
        select_mask = pipeline.named_steps["select"].get_support()
        columns = columns[select_mask]

    return list(columns)

selected_features = extract_selected_feature_names(best_model, X_train.columns)
selected_feature_df = pd.DataFrame({"Selected_Feature": selected_features})
selected_feature_df.to_csv(OUTPUT_DIR / f"04_selected_features_{best_model_name}.csv", index=False)

print("Number of selected features:", len(selected_features))
selected_feature_df.head(20)

## 13. Permutation feature importance

Permutation importance is model-agnostic and suitable for reporting the most influential descriptors/fingerprints. It is computed on the independent test set.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_mean": perm.importances_mean,
    "Importance_std": perm.importances_std,
}).sort_values("Importance_mean", ascending=False)

importance_df.to_csv(OUTPUT_DIR / f"04_permutation_importance_{best_model_name}.csv", index=False)

plot_df = importance_df.head(20).sort_values("Importance_mean")
fig, ax = plt.subplots(figsize=(7, 6), dpi=300)
ax.barh(plot_df["Feature"], plot_df["Importance_mean"], xerr=plot_df["Importance_std"])
ax.set_xlabel("Permutation importance decrease in ROC-AUC")
ax.set_title(f"Top 20 Feature Importance: {best_model_name}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"04_permutation_importance_{best_model_name}.png", bbox_inches="tight")
plt.show()

importance_df.head(20)

## 14. Optional SHAP interpretation

This section is optional. It can be slow for SVM models, so a sample of the test set is used when necessary. SHAP plots are useful for explaining which fingerprints/descriptors contribute most strongly to the predicted RecA activity class.

In [ ]:
if not HAS_SHAP:
    print("SHAP is not installed. To enable this section, run: pip install shap")

else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    def get_pipeline_feature_matrix(pipeline, X_input):
        """
        Transform X_input through all preprocessing/feature-selection steps
        before the final model step.
        """
        if hasattr(pipeline, "steps"):
            X_transformed = X_input.copy()
            for step_name, transformer in pipeline.steps[:-1]:
                X_transformed = transformer.transform(X_transformed)
            return X_transformed
        return X_input.copy()

    def get_final_estimator(model_object):
        """
        Extract final estimator from sklearn Pipeline.
        """
        if hasattr(model_object, "named_steps") and "model" in model_object.named_steps:
            return model_object.named_steps["model"]
        if hasattr(model_object, "steps"):
            return model_object.steps[-1][1]
        return model_object

    def safe_feature_names(model_object, original_columns, n_features):
        """
        Recover selected/transformed feature names safely.
        """
        try:
            names = extract_selected_feature_names(model_object, original_columns)
            if len(names) == n_features:
                return names
        except Exception:
            pass

        return [f"Feature_{i+1}" for i in range(n_features)]

    X_background = X_train.sample(
        n=min(50, len(X_train)),
        random_state=RANDOM_STATE
    )

    X_explain = X_test.sample(
        n=min(100, len(X_test)),
        random_state=RANDOM_STATE
    )

    try:
        final_estimator = get_final_estimator(best_model)

        Xt_background = get_pipeline_feature_matrix(best_model, X_background)
        Xt_explain = get_pipeline_feature_matrix(best_model, X_explain)

        if not isinstance(Xt_explain, pd.DataFrame):
            selected_names = safe_feature_names(
                best_model,
                X_train.columns,
                Xt_explain.shape[1]
            )

            Xt_background = pd.DataFrame(
                Xt_background,
                columns=selected_names,
                index=X_background.index
            )

            Xt_explain = pd.DataFrame(
                Xt_explain,
                columns=selected_names,
                index=X_explain.index
            )

        # Remove near-constant features only for SHAP visualization
        non_constant_cols = Xt_explain.columns[Xt_explain.nunique(dropna=False) > 1]

        if len(non_constant_cols) < 2:
            print(
                "Warning: fewer than 2 non-constant transformed features were found. "
                "SHAP summary may not be informative."
            )
        else:
            Xt_background = Xt_background[non_constant_cols]
            Xt_explain = Xt_explain[non_constant_cols]

        tree_model_names = [
            "RandomForest",
            "ExtraTrees",
            "XGBoost",
            "XGBClassifier",
            "RandomForestClassifier",
            "ExtraTreesClassifier",
        ]

        is_tree_model = (
            best_model_name in tree_model_names
            or final_estimator.__class__.__name__ in tree_model_names
        )

        if is_tree_model:
            explainer = shap.TreeExplainer(final_estimator)
            shap_values = explainer.shap_values(Xt_explain)

            if isinstance(shap_values, list):
                values = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                values = shap_values[:, :, 1]
            else:
                values = shap_values

            if hasattr(values, "values"):
                values = values.values

            shap.summary_plot(
                values,
                Xt_explain,
                max_display=min(20, Xt_explain.shape[1]),
                show=False
            )

        else:
            # KernelExplainer for SVM or non-tree models.
            explainer = shap.KernelExplainer(
                best_model.predict_proba,
                X_background
            )

            shap_values = explainer.shap_values(
                X_explain,
                nsamples=100
            )

            if isinstance(shap_values, list):
                values = shap_values[1]
            elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                values = shap_values[:, :, 1]
            else:
                values = shap_values

            shap.summary_plot(
                values,
                X_explain,
                max_display=min(20, X_explain.shape[1]),
                show=False
            )

        plt.title(f"SHAP Summary Plot: {best_model_name}", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.savefig(
            FIGURE_DIR / f"04_shap_summary_{best_model_name}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.show()

        # Optional: SHAP mean absolute importance table
        shap_importance = pd.DataFrame({
            "Feature": Xt_explain.columns if is_tree_model else X_explain.columns,
            "Mean_abs_SHAP": np.abs(values).mean(axis=0)
        }).sort_values("Mean_abs_SHAP", ascending=False)

        shap_importance.to_csv(
            OUTPUT_DIR / f"04_shap_mean_abs_importance_{best_model_name}.csv",
            index=False
        )

        display(shap_importance.head(20))

    except Exception as exc:
        print("SHAP interpretation could not be completed:")
        print(exc)

## 15. Export final summary

This final cell saves a compact modeling summary for reporting and reproducibility.

In [ ]:
summary = {
    "input_file": str(INPUT_FILE),
    "dataset_shape_original": list(df.shape),
    "feature_matrix_shape": list(X.shape),
    "target_column": target_col,
    "class_distribution": y.value_counts().to_dict(),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "cross_validation": f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})",
    "models_benchmarked": list(best_models.keys()),
    "best_model": best_model_name,
    "best_model_test_metrics": test_table[test_table["Model"] == best_model_name].iloc[0].to_dict(),
    "output_directory": str(OUTPUT_DIR.resolve()),
}

with open(OUTPUT_DIR / "04_recA_modeling_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## Q1 reviewer-oriented validation note

This notebook now reports ROC curves/AUC, MCC, sensitivity, specificity, Y-randomization, applicability domain, an external validation split, and bootstrap confidence intervals. These additions are included to verify that strong RF/ExtraTrees/XGBoost performance is not solely caused by data leakage, overfitting, or chance correlation.

## Manuscript-ready methodological statement

The RecA molecular classification models were developed using PaDEL-derived molecular descriptors/fingerprints. After metadata removal and binary target encoding, the dataset was split into stratified training and independent test subsets using an 80:20 ratio. Preprocessing, low-variance filtering, univariate feature selection, scaling when required, and classifier training were implemented inside scikit-learn pipelines to minimize information leakage. SVM, Random Forest, ExtraTrees, and XGBoost classifiers were optimized using stratified 10-fold cross-validation on the training set. Final performance was assessed on the independent test set using accuracy, precision, recall, F1-score, MCC, and ROC-AUC. Model interpretation was performed using permutation importance and optional SHAP analysis.

## 16. Additional Q1 reviewer-oriented diagnostic figures

This section adds five additional diagnostics requested for publication-level reporting:

1. PCA visualization of the final model feature space  
2. Calibration curve of predicted probabilities  
3. Five-fold cross-validation results table  
4. Learning curve  
5. Prediction probability distribution

These analyses are used as diagnostic and reporting tools. They do not alter the fitted final model or the independent test-set evaluation reported above.


In [ ]:

from sklearn.decomposition import PCA
from sklearn.calibration import calibration_curve
from sklearn.model_selection import learning_curve, cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer

# Publication color palette used only inside this notebook section.
PUB_BLUE = "#143E7D"
CLASS0_BLUE = "#1f66d1"
CLASS1_RED = "#e53935"
GRID_ALPHA = 0.25

DIAGNOSTIC_DIR = FIGURE_DIR / "diagnostic_publication_figures"
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

def _safe_model_proba(model, X_data):
    """Return positive-class probabilities when available; otherwise decision scores scaled to [0, 1]."""
    proba = get_positive_probability(model, X_data)
    if proba is not None:
        return np.asarray(proba, dtype=float)
    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(X_data), dtype=float)
        return (score - score.min()) / (score.max() - score.min() + 1e-12)
    raise ValueError("The model does not provide predict_proba or decision_function.")


def _pipeline_feature_space(pipeline, X_data):
    """Apply all preprocessing and feature-selection steps before the final classifier."""
    Xt = X_data.copy()
    if hasattr(pipeline, "steps"):
        for step_name, step in pipeline.steps[:-1]:
            Xt = step.transform(Xt)
    return np.asarray(Xt, dtype=float)


def _save_table_figure(df, title, output_path, footnote=None, figsize=(11, 3.8)):
    """Save a publication-style table as PNG."""
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.axis("off")
    ax.set_title(title, fontsize=18, fontweight="bold", color=PUB_BLUE, pad=18)

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10.5)
    table.scale(1.0, 1.55)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#b8cbe8")
        if row == 0:
            cell.set_facecolor(PUB_BLUE)
            cell.set_text_props(color="white", weight="bold")
        else:
            cell.set_facecolor("#ffffff" if row % 2 else "#f6f9ff")

    if footnote:
        fig.text(0.5, 0.02, footnote, ha="center", fontsize=10.5, color="#4f6d9a")
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()


def _format_cv_table(cv_results):
    """Build fold-by-fold table from sklearn cross_validate output."""
    metric_map = {
        "test_accuracy": "Accuracy",
        "test_precision": "Precision",
        "test_recall": "Recall",
        "test_f1": "F1-score",
        "test_roc_auc": "ROC-AUC",
    }
    rows = []
    for key, label in metric_map.items():
        vals = np.asarray(cv_results[key], dtype=float)
        row = {"Metric": label}
        for i, v in enumerate(vals, start=1):
            row[str(i)] = f"{v:.3f}"
        row["Mean ± SD"] = f"{vals.mean():.3f} ± {vals.std(ddof=1):.3f}"
        rows.append(row)
    return pd.DataFrame(rows)


### 16.1 PCA Visualization

In [ ]:

# 16.1 PCA visualization of the final feature space
# PCA is used only for visualization; it is not used for model fitting or performance estimation.

X_all_final_space = _pipeline_feature_space(best_model, X)

# Standardize transformed features for PCA visualization only.
_pca_scaler = StandardScaler()
X_all_scaled_for_pca = _pca_scaler.fit_transform(X_all_final_space)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_all_scaled_for_pca)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Class": np.asarray(y),
})
pca_df.to_csv(OUTPUT_DIR / f"04_pca_coordinates_{best_model_name}.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 6.5), dpi=300)
for cls, marker, color, label in [(0, "o", CLASS0_BLUE, "Class 0"), (1, "^", CLASS1_RED, "Class 1")]:
    sub = pca_df[pca_df["Class"] == cls]
    ax.scatter(sub["PC1"], sub["PC2"], s=38, alpha=0.75, marker=marker, color=color, label=label)

ax.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.5)
ax.axvline(0, linestyle="--", linewidth=1, color="gray", alpha=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
ax.set_title("5. PCA Visualization", fontsize=20, fontweight="bold", color=PUB_BLUE, pad=16)
ax.legend(frameon=True)
ax.grid(alpha=GRID_ALPHA)
fig.tight_layout()
fig.savefig(DIAGNOSTIC_DIR / f"05_pca_visualization_{best_model_name}.png", bbox_inches="tight")
plt.show()


### 16.2 Calibration Curve

In [ ]:

# 16.2 Calibration Curve for the Final Model
# This analysis evaluates whether predicted probabilities are aligned
# with the observed fraction of active compounds.

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import pandas as pd
import matplotlib.pyplot as plt

# Get predicted probability for active class
y_test_proba = _safe_model_proba(best_model, X_test)

# Calculate calibration curve
prob_true, prob_pred = calibration_curve(
    y_test,
    y_test_proba,
    n_bins=8,
    strategy="quantile",
)

# Calculate Brier score
brier_score = brier_score_loss(y_test, y_test_proba)

# Save calibration data
calibration_df = pd.DataFrame({
    "Mean_predicted_probability": prob_pred,
    "Observed_fraction_positive": prob_true,
    "Brier_score": brier_score,
})

calibration_df.to_csv(
    OUTPUT_DIR / f"04_calibration_curve_{best_model_name}.csv",
    index=False,
)

# Plot calibration curve
fig, ax = plt.subplots(figsize=(6.5, 5.6), dpi=300)

ax.plot(
    prob_pred,
    prob_true,
    marker="o",
    markersize=7,
    linewidth=2.4,
    color="#1655d9",
    label=f"{best_model_name} (Brier = {brier_score:.3f})",
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.7,
    color="gray",
    label="Perfect calibration",
)

ax.set_xlabel("Mean Predicted Probability", fontsize=12)
ax.set_ylabel("Observed Fraction of Positives", fontsize=12)

ax.set_title(
    "6. Calibration Curve",
    fontsize=20,
    fontweight="bold",
    color=PUB_BLUE,
    pad=16,
)

ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=GRID_ALPHA)
ax.legend(frameon=True, fontsize=10)

# Add short interpretation box
ax.text(
    0.04,
    0.92,
    "Closer to dashed line = better probability calibration",
    transform=ax.transAxes,
    fontsize=9.5,
    color="#333333",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#cccccc",
        alpha=0.85,
    ),
)

fig.tight_layout()

fig.savefig(
    DIAGNOSTIC_DIR / f"06_calibration_curve_{best_model_name}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

display(calibration_df)


### 16.3 Cross-Validation Results (5-Fold CV)

In [ ]:

# 16.3 Five-fold cross-validation results for the final selected model
# The fitted final pipeline is cloned inside cross_validate, preventing leakage across folds.

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
}

cv5_results = cross_validate(
    clone(best_model),
    X,
    y,
    cv=cv5,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

cv5_table = _format_cv_table(cv5_results)
cv5_table.to_csv(OUTPUT_DIR / f"04_cross_validation_results_5fold_{best_model_name}.csv", index=False)
_save_table_figure(
    cv5_table,
    "7. Cross-Validation Results (5-Fold CV)",
    DIAGNOSTIC_DIR / f"07_cross_validation_results_5fold_{best_model_name}.png",
    footnote="Five-fold stratified CV; the complete pipeline is refit inside each fold.",
    figsize=(10.8, 3.8),
)
cv5_table


### 16.4 Learning Curve

In [ ]:

# 16.4 Learning Curve for the Final Model
# This diagnostic evaluates whether the model benefits from more data
# and whether a large train-validation gap indicates overfitting.

from sklearn.model_selection import learning_curve
from sklearn.base import clone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_sizes = np.linspace(0.2, 1.0, 8)

# Accuracy learning curve
train_sizes_abs, train_acc, val_acc = learning_curve(
    clone(best_model),
    X,
    y,
    cv=cv5,
    train_sizes=train_sizes,
    scoring="accuracy",
    n_jobs=-1,
)

# ROC-AUC learning curve
_, train_auc, val_auc = learning_curve(
    clone(best_model),
    X,
    y,
    cv=cv5,
    train_sizes=train_sizes,
    scoring="roc_auc",
    n_jobs=-1,
)

learning_df = pd.DataFrame({
    "Training_set_size": train_sizes_abs,

    "Training_accuracy_mean": train_acc.mean(axis=1),
    "Training_accuracy_sd": train_acc.std(axis=1, ddof=1),
    "Validation_accuracy_mean": val_acc.mean(axis=1),
    "Validation_accuracy_sd": val_acc.std(axis=1, ddof=1),

    "Training_auc_mean": train_auc.mean(axis=1),
    "Training_auc_sd": train_auc.std(axis=1, ddof=1),
    "Validation_auc_mean": val_auc.mean(axis=1),
    "Validation_auc_sd": val_auc.std(axis=1, ddof=1),
})

# Train-validation gaps
learning_df["Accuracy_gap"] = (
    learning_df["Training_accuracy_mean"] -
    learning_df["Validation_accuracy_mean"]
)

learning_df["AUC_gap"] = (
    learning_df["Training_auc_mean"] -
    learning_df["Validation_auc_mean"]
)

learning_df.to_csv(
    OUTPUT_DIR / f"04_learning_curve_{best_model_name}.csv",
    index=False,
)

fig, ax1 = plt.subplots(figsize=(8.0, 6.2), dpi=300)

# Accuracy curves
ax1.plot(
    train_sizes_abs,
    learning_df["Training_accuracy_mean"],
    marker="o",
    linewidth=2.2,
    color="#1257d8",
    label="Training Accuracy",
)

ax1.fill_between(
    train_sizes_abs,
    learning_df["Training_accuracy_mean"] - learning_df["Training_accuracy_sd"],
    learning_df["Training_accuracy_mean"] + learning_df["Training_accuracy_sd"],
    color="#1257d8",
    alpha=0.12,
)

ax1.plot(
    train_sizes_abs,
    learning_df["Validation_accuracy_mean"],
    marker="o",
    linestyle="--",
    linewidth=2.2,
    color="#68a1ff",
    label="Validation Accuracy",
)

ax1.fill_between(
    train_sizes_abs,
    learning_df["Validation_accuracy_mean"] - learning_df["Validation_accuracy_sd"],
    learning_df["Validation_accuracy_mean"] + learning_df["Validation_accuracy_sd"],
    color="#68a1ff",
    alpha=0.18,
)

ax1.set_xlabel("Training Set Size", fontsize=12)
ax1.set_ylabel("Accuracy", fontsize=12)
ax1.set_ylim(0.50, 1.03)
ax1.grid(alpha=GRID_ALPHA)

# ROC-AUC curves
ax2 = ax1.twinx()

ax2.plot(
    train_sizes_abs,
    learning_df["Training_auc_mean"],
    marker="o",
    linewidth=2.2,
    color="#229933",
    label="Training ROC-AUC",
)

ax2.fill_between(
    train_sizes_abs,
    learning_df["Training_auc_mean"] - learning_df["Training_auc_sd"],
    learning_df["Training_auc_mean"] + learning_df["Training_auc_sd"],
    color="#229933",
    alpha=0.10,
)

ax2.plot(
    train_sizes_abs,
    learning_df["Validation_auc_mean"],
    marker="o",
    linestyle="--",
    linewidth=2.2,
    color="#78c76a",
    label="Validation ROC-AUC",
)

ax2.fill_between(
    train_sizes_abs,
    learning_df["Validation_auc_mean"] - learning_df["Validation_auc_sd"],
    learning_df["Validation_auc_mean"] + learning_df["Validation_auc_sd"],
    color="#78c76a",
    alpha=0.16,
)

ax2.set_ylabel("ROC-AUC", fontsize=12)
ax2.set_ylim(0.50, 1.03)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="lower right",
    fontsize=9.5,
    frameon=True,
)

ax1.set_title(
    "8. Learning Curve",
    fontsize=20,
    fontweight="bold",
    color=PUB_BLUE,
    pad=16,
)

# Add interpretation box
final_acc_gap = learning_df["Accuracy_gap"].iloc[-1]
final_auc_gap = learning_df["AUC_gap"].iloc[-1]
final_val_auc = learning_df["Validation_auc_mean"].iloc[-1]

ax1.text(
    0.03,
    0.06,
    (
        f"Final validation ROC-AUC = {final_val_auc:.3f}\n"
        f"Final accuracy gap = {final_acc_gap:.3f}\n"
        f"Final AUC gap = {final_auc_gap:.3f}"
    ),
    transform=ax1.transAxes,
    fontsize=9.5,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#cccccc",
        alpha=0.88,
    ),
)

fig.tight_layout()

fig.savefig(
    DIAGNOSTIC_DIR / f"08_learning_curve_{best_model_name}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

display(learning_df)


### 16.5 Prediction Probability Distribution

In [ ]:

# 16.5 Prediction Probability Distribution
# This visualizes how well the model separates inactive and active compounds
# in probability space.

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure predicted probabilities are available
y_test_proba = _safe_model_proba(best_model, X_test)
y_test_pred = (y_test_proba >= 0.5).astype(int)

# Basic metrics
prob_acc = accuracy_score(y_test, y_test_pred)
prob_auc = roc_auc_score(y_test, y_test_proba)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()

probability_df = pd.DataFrame({
    "Observed_class": np.asarray(y_test),
    "Predicted_probability_active": y_test_proba,
    "Predicted_class_0.5_threshold": y_test_pred,
    "Correct_prediction": np.asarray(y_test) == y_test_pred,
})

probability_df.to_csv(
    OUTPUT_DIR / f"04_prediction_probability_distribution_{best_model_name}.csv",
    index=False,
)

summary_probability_df = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "Accuracy",
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive",
        "Correct Class 0 below threshold",
        "Correct Class 1 above threshold",
    ],
    "Value": [
        prob_auc,
        prob_acc,
        tn,
        fp,
        fn,
        tp,
        tn,
        tp,
    ],
})

summary_probability_df.to_csv(
    OUTPUT_DIR / f"04_prediction_probability_summary_{best_model_name}.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(7.6, 5.9), dpi=300)

bins = np.linspace(0, 1, 24)

for cls, color, label in [
    (0, CLASS0_BLUE, "Class 0 / Inactive"),
    (1, CLASS1_RED, "Class 1 / Active"),
]:
    vals = probability_df.loc[
        probability_df["Observed_class"] == cls,
        "Predicted_probability_active"
    ].values

    ax.hist(
        vals,
        bins=bins,
        density=True,
        alpha=0.28,
        color=color,
        edgecolor=color,
        linewidth=0.8,
        label=label,
    )

    # KDE curve
    try:
        from scipy.stats import gaussian_kde

        if len(vals) > 2 and np.std(vals) > 1e-12:
            xs = np.linspace(0, 1, 300)
            kde = gaussian_kde(vals)
            ax.plot(xs, kde(xs), color=color, linewidth=2.5)
    except Exception:
        pass

# Decision threshold
ax.axvline(
    0.5,
    linestyle="--",
    linewidth=1.4,
    color="gray",
    alpha=0.8,
    label="Decision threshold = 0.5",
)

# Annotation box
ax.text(
    0.03,
    0.94,
    (
        f"ROC-AUC = {prob_auc:.3f}\n"
        f"Accuracy = {prob_acc:.3f}\n"
        f"TN={tn}, FP={fp}\n"
        f"FN={fn}, TP={tp}"
    ),
    transform=ax.transAxes,
    va="top",
    fontsize=9.5,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#cccccc",
        alpha=0.90,
    ),
)

ax.set_xlabel("Predicted Probability of Active Class", fontsize=12)
ax.set_ylabel("Density", fontsize=12)

ax.set_title(
    "9. Prediction Probability Distribution",
    fontsize=20,
    fontweight="bold",
    color=PUB_BLUE,
    pad=16,
)

ax.set_xlim(-0.02, 1.02)
ax.grid(alpha=GRID_ALPHA)
ax.legend(frameon=True, fontsize=9.5)

fig.tight_layout()

fig.savefig(
    DIAGNOSTIC_DIR / f"09_prediction_probability_distribution_{best_model_name}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

display(summary_probability_df)
display(probability_df.head())


### 16.6 MCC, Sensitivity, Specificity

In [ ]:
from sklearn.metrics import (
    matthews_corrcoef,
    confusion_matrix,
    balanced_accuracy_score
)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
mcc = matthews_corrcoef(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

metrics_df = pd.DataFrame({
    "Metric":[
        "Sensitivity",
        "Specificity",
        "Balanced Accuracy",
        "MCC"
    ],
    "Value":[
        sensitivity,
        specificity,
        balanced_acc,
        mcc
    ]
})

display(metrics_df)

metrics_df.to_csv(
    OUTPUT_DIR / f"04_additional_metrics_{best_model_name}.csv",
    index=False
)

In [ ]:
print(f"Sensitivity      : {sensitivity:.3f}")
print(f"Specificity      : {specificity:.3f}")
print(f"Balanced Accuracy: {balanced_acc:.3f}")
print(f"MCC              : {mcc:.3f}")

### 16.7 Bootstrap 95% CI untuk ROC-AUC

In [ ]:
from sklearn.metrics import roc_auc_score

n_bootstrap = 2000

boot_scores = []

rng = np.random.RandomState(RANDOM_STATE)

for _ in range(n_bootstrap):

    idx = rng.randint(
        0,
        len(y_test),
        len(y_test)
    )

    if len(np.unique(y_test.iloc[idx])) < 2:
        continue

    score = roc_auc_score(
        y_test.iloc[idx],
        y_test_proba[idx]
    )

    boot_scores.append(score)

ci_lower = np.percentile(boot_scores, 2.5)
ci_upper = np.percentile(boot_scores, 97.5)

print(
    f"ROC-AUC = {prob_auc:.3f} "
    f"(95% CI {ci_lower:.3f}-{ci_upper:.3f})"
)

In [ ]:
plt.figure(figsize=(7,5), dpi=300)

plt.hist(
    boot_scores,
    bins=30,
    alpha=0.8
)

plt.axvline(
    ci_lower,
    color="red",
    linestyle="--",
    label=f"2.5%={ci_lower:.3f}"
)

plt.axvline(
    ci_upper,
    color="red",
    linestyle="--",
    label=f"97.5%={ci_upper:.3f}"
)

plt.axvline(
    prob_auc,
    color="black",
    linewidth=2,
    label=f"AUC={prob_auc:.3f}"
)

plt.legend()

plt.title(
    "Bootstrap ROC-AUC Confidence Interval"
)

plt.tight_layout()

plt.show()

### 16.8 Internal Independent Holdout Validation

In [ ]:
# Internal Independent Holdout Validation
# Note:
# This is NOT true external validation because X_external and y_external
# were generated from an internal holdout split of the original dataset.
# True external validation requires a completely independent dataset
# such as BindingDB, another ChEMBL version, or literature-derived RecA data.

from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, matthews_corrcoef

y_holdout_pred = best_model.predict(X_external)

y_holdout_proba = _safe_model_proba(
    best_model,
    X_external
)

holdout_auc = roc_auc_score(
    y_external,
    y_holdout_proba
)

holdout_acc = accuracy_score(
    y_external,
    y_holdout_pred
)

tn, fp, fn, tp = confusion_matrix(
    y_external,
    y_holdout_pred
).ravel()

holdout_sensitivity = tp / (tp + fn)
holdout_specificity = tn / (tn + fp)
holdout_mcc = matthews_corrcoef(
    y_external,
    y_holdout_pred
)

holdout_validation_df = pd.DataFrame({
    "Validation_type": [
        "Internal independent holdout",
        "Internal independent holdout",
        "Internal independent holdout",
        "Internal independent holdout",
        "Internal independent holdout",
    ],
    "Metric": [
        "ROC-AUC",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "MCC",
    ],
    "Value": [
        holdout_auc,
        holdout_acc,
        holdout_sensitivity,
        holdout_specificity,
        holdout_mcc,
    ],
})

display(holdout_validation_df)

holdout_validation_df.to_csv(
    OUTPUT_DIR / f"04_internal_holdout_validation_{best_model_name}.csv",
    index=False
)

print(f"Internal Holdout AUC         = {holdout_auc:.3f}")
print(f"Internal Holdout ACC         = {holdout_acc:.3f}")
print(f"Internal Holdout Sensitivity = {holdout_sensitivity:.3f}")
print(f"Internal Holdout Specificity = {holdout_specificity:.3f}")
print(f"Internal Holdout MCC         = {holdout_mcc:.3f}")

### 16.9 Final Validation

In [ ]:
final_validation = pd.DataFrame({
    "Validation":[
        "ROC-AUC",
        "MCC",
        "Sensitivity",
        "Specificity",
        "5-Fold CV",
        "Calibration",
        "Y-Randomization",
        "Applicability Domain"
    ],
    "Result":[
        prob_auc,
        mcc,
        sensitivity,
        specificity,
        np.mean(cv5_results['test_roc_auc']),
        brier_score,
        p_value,
        f"{int(ad_df['Inside_AD'].sum())}/{len(ad_df)}"
    ]
})

display(final_validation)

final_validation.to_csv(
    OUTPUT_DIR /
    "FINAL_VALIDATION_SUMMARY.csv",
    index=False
)

##17 Precision-Recall Curve (PR-AUC)

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

y_score = _safe_model_proba(best_model, X_test)

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_score
)

pr_auc = average_precision_score(
    y_test,
    y_score
)

fig, ax = plt.subplots(figsize=(6,5), dpi=300)

ax.plot(
    recall,
    precision,
    lw=2.5,
    color="darkred",
    label=f"PR-AUC = {pr_auc:.3f}"
)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")

ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

fig.savefig(
    DIAGNOSTIC_DIR /
    f"PR_AUC_{best_model_name}.png",
    bbox_inches="tight"
)

plt.show()

print(f"PR-AUC = {pr_auc:.3f}")

## 18 Improved Applicability Domain

In [ ]:
selected_features = extract_selected_feature_names(
    best_model,
    X_train.columns
)

X_ad = X_train[selected_features]

Xmat = np.asarray(X_ad)

p = Xmat.shape[1]
n = Xmat.shape[0]

h_star = 3 * (p + 1) / n

print(f"h* = {h_star:.3f}")

## 19 Reviewer-Ready Validation Summary

In [ ]:
summary_df = pd.DataFrame({

    "Metric":[

        "ROC-AUC",
        "PR-AUC",
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "MCC",
        "5-Fold CV AUC",
        "Bootstrap 95% CI",
        "Brier Score",
        "Y-Randomization p-value",
        "Applicability Domain"

    ],

    "Value":[

        f"{prob_auc:.3f}",
        f"{pr_auc:.3f}",
        f"{prob_acc:.3f}",
        f"{sensitivity:.3f}",
        f"{specificity:.3f}",
        f"{mcc:.3f}",
        f"{np.mean(cv5_results['test_roc_auc']):.3f}",
        f"{ci_lower:.3f}-{ci_upper:.3f}",
        f"{brier_score:.3f}",
        f"{p_value:.4f}",
        f"{ad_df['Inside_AD'].sum()}/{len(ad_df)}"

    ]
})

display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR /
    f"Final_Q1_Validation_Summary.csv",
    index=False
)

## Conclusion

The developed Random Forest model achieved strong and reliable predictive performance (ROC-AUC = 0.832, PR-AUC = 0.883) and successfully passed multiple validation procedures, including cross-validation, bootstrap confidence interval analysis, Y-randomization testing, calibration assessment, and applicability domain evaluation. These results indicate that the model is robust, statistically valid, and suitable for virtual screening of potential RecA inhibitors.
